[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Feellived/tobigs-26th-stat-inference-regression/blob/main/assignment/assignment.ipynb)

# 투빅스 26기 1주차 2교시 — 통계적 추론 + 회귀 과제  ·  Framingham 심장연구

## 과제 목적

이번 과제는 **강의에서 관통한 바로 그 데이터, Framingham 심장연구(참가자 4,240명)** 를 여러분이 직접 손에 쥐고 로드 → EDA → 추론 → 회귀 → 해석/예측까지 관통하는 프로젝트입니다. 강의가 이 데이터로 개념을 '보여주는' 쪽이었다면, 과제는 여러분이 같은 데이터로 그 개념을 '직접 만들어 보는' 쪽입니다.

큰 그림을 먼저 잡아 둡시다. Framingham 연구는 '위험인자(risk factor)'라는 개념을 만든 곳이고, 통계적 추론과 회귀는 곧 **그 위험인자를 발굴하는 절차**입니다 — **절차① 검정**(어떤 변수가 발병군에서 다른가)과 **절차② 회귀**(그 차이가 얼마나 큰가, 통제해도 남는가). 그리고 이 연구가 임상에 실제로 남긴 최종 산물은 **"앞으로 10년 안에 심장질환이 생길 확률"을 개인별로 계산하는 Framingham Risk Score**였습니다. 강의는 수축기혈압 `sysBP`(연속) 회귀에서 멈췄지만, **이 과제는 그 뒤를 이어 진짜 표적 `TenYearCHD`(0/1)를 로지스틱 회귀로 예측하는 데까지 스토리를 완성**합니다.

주인공은 수축기혈압 `sysBP`(연속)와 10년 내 관상동맥심질환 `TenYearCHD`(0/1)입니다. 강의에서 본 숫자 — 평균 혈압의 신뢰구간, 심장병군 vs 비심장병군의 혈압 차이 검정, 나이→혈압 회귀, 다중회귀와 공선성 — 을 **여러분 손으로 재현·해석**하고, 마지막에 **로지스틱 회귀로 발병 위험을 예측(= Risk Score의 엔진)** 하는 데까지 갑니다. 장난감 데이터와 달리 결측·이진변수·다변수가 있어 조금 더 까다롭지만, 그 까다로움이 곧 실전 감각입니다.

각 코드 셀의 빈칸(`# TODO: ____`, `____`)을 채우면서 "지금 나는 어느 단계에서 무엇을 하고 있는가"를 계속 확인하세요. **맨 위 셀부터 순서대로** 실행하면 앞 단계의 결과(변수)가 뒤 단계로 이어집니다.

## 채점 · AI 활용 안내

- **실행은 기본점, 사고가 변별점입니다.** 코드를 돌려 값을 얻는 것만으로는 부족하고, 특히 §2 "스스로 생각하기"에서 본인이 관찰한 값으로 '왜 그런가'를 설명해야 점수가 갈립니다.
- **답이 본인 시드·실행값에 묶여 있습니다.** `MY_SEED`가 사람마다 달라 부트스트랩 신뢰구간·CLT 시뮬레이션·train/test 분할 결과가 **학생마다 다릅니다.** AI가 내놓는 일반적인 숫자를 그대로 옮기면 **본인 실행값과 어긋나 감점**됩니다. 반드시 본인 노트북에서 나온 값을 인용하세요.
- **AI(GPT 등)는 검산기·설명 도우미로만.** 결과가 맞는지 판단·검증·해석하는 것은 본인이 합니다. §5에 실제 대화 내역과 함께, AI가 준 답 중 틀렸거나 이 데이터엔 안 맞았던 것을 어떻게 잡아냈는지 정리하세요.

## 0. 환경 설정 — 라이브러리, 시드, 데이터 로드

이번 과제는 **Framingham 심장연구(Framingham Heart Study)** 데이터를 씁니다. 1948년 미국 매사추세츠주 프레이밍햄 주민들을 수십 년간 추적한 역학 코호트로, "심혈관 질환 위험요인"이라는 개념 자체가 여기서 나왔습니다. 우리가 쓰는 표는 참가자 **4,240명**에 대해 성별·나이·흡연·혈압·콜레스테롤 등 16개 변수를 담고 있고, 마지막 열 `TenYearCHD`가 **10년 안에 관상동맥심질환(CHD)이 발생했는지(1/0)** 를 알려줍니다.

강의에서 다룬 흐름(EDA → 통계적 추론·검정 → 회귀)을 이 한 데이터로 처음부터 끝까지 관통합니다. 우리의 주인공 변수는 **`sysBP`(수축기 혈압, systolic blood pressure)** 입니다.

이 셀에서 하는 일은 세 가지입니다.

**1) 라이브러리 import.** 앞으로 모든 셀이 여기서 불러온 이름(`np`, `pd`, `smf` 등)을 그대로 씁니다. 한 번만 실행하면 노트북 전체에서 재사용됩니다.

**2) `MY_SEED` 설정.** *시드(seed)* 는 난수 생성기의 출발점입니다. 컴퓨터의 "무작위"는 사실 시작값 하나로 정해지는 재현 가능한 수열이라, 같은 시드를 주면 몇 번을 돌려도 **똑같은 난수**가 나옵니다. 뒤에서 CLT 시뮬레이션·부트스트랩·train/test 분할처럼 무작위가 개입하는 단계가 나오는데, 시드를 고정해야 여러분의 결과와 정답의 숫자가 재현되고 채점도 가능합니다. 여기서는 **본인 학번 뒷 4자리**를 시드로 쓰겠습니다(예: 학번이 2026-12345면 `2345`).

**3) 데이터 로드.** 공개 URL에서 먼저 읽고, 실패하면(오프라인·URL 접근 불가) 같은 폴더에 업로드해 둔 `framingham.csv`로 대체합니다. 이렇게 `try/except`로 감싸두면 어느 환경에서도 셀이 멈추지 않습니다.

> 시드를 고정한다고 "결과가 편향된다"거나 "정답에 맞춘다"는 뜻이 아닙니다. 무작위 절차 자체는 그대로이고, 다만 **재현 가능하게** 만들 뿐입니다. 시드를 바꾸면 숫자는 조금씩 달라지지만 결론(예: 신뢰구간의 폭, 검정의 유의성)은 사실상 같아야 정상입니다.

In [ ]:
# ── 라이브러리 import (한 번만 실행; 이후 모든 셀이 이 이름을 그대로 사용) ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, roc_curve

# ── 0. 환경 설정 ──────────────────────────────────────────────
# (1) 라이브러리 import — 이후 모든 셀이 여기서 불러온 이름을 그대로 사용

# 그래프 기본 스타일(선택)
sns.set_theme(style="whitegrid")

# (2) 시드 설정 — 본인 학번 뒷 4자리를 넣으세요 (예: 2026-12345 → 2345)
MY_SEED = ____              # TODO: 학번 뒷 4자리
np.random.seed(____)        # TODO: MY_SEED로 난수 생성기 고정

# (3) 데이터 로드 — 공개 URL 우선, 실패 시 업로드한 로컬 파일로 대체
# 데이터 로드: 공개 URL에서 바로 읽습니다(업로드 불필요). 실패 시 업로드한 로컬 파일 사용.
DATA_URL = "https://raw.githubusercontent.com/GauravPadawe/Framingham-Heart-Study/master/framingham.csv"
try:
    df = pd.read_csv(DATA_URL)
except Exception:
    df = pd.read_csv("framingham.csv")   # Colab: 좌측 파일 탭에 framingham.csv 업로드 후

# 이후 섹션 전체가 공유하는 수치형 열 목록
num_cols = ["age", "sysBP", "diaBP", "totChol", "BMI", "glucose", "heartRate"]

# 로드 확인
print("데이터 크기 (행, 열):", df.shape)          # 기대: (4240, 16)
print("MY_SEED =", MY_SEED)
print("\n[첫 3행]")
print(df.head(3))
print("\n[열별 결측치 개수 (0이 아닌 것만)]")
print(df.isna().sum()[df.isna().sum() > 0])       # TODO: 결측이 어느 열에 있는지 확인


**Q0. 데이터를 제대로 불렀는지 확인해 봅시다.**

1. 출력된 `df.shape`는 무엇인가요? 행(참가자) 수와 열(변수) 수를 **본인이 본 실제 숫자**로 적으세요.
2. 결측치(NaN)가 있는 열 중 **결측이 가장 많은 열**은 무엇이고 몇 개인가요? (tips 데이터와 달리 Framingham은 결측이 있어서, 뒤 섹션에서 해당 열을 다룰 때 `dropna()` 같은 처리가 필요합니다. 어떤 열이 문제가 될지 미리 눈에 익혀 두세요.)
3. `MY_SEED`를 다른 값으로 바꿔서 이 셀을 다시 실행하면 `df.shape`가 달라질까요? 왜 그런지 한 줄로 설명해 보세요.

> 힌트(3번): 시드는 *무작위 절차*(시뮬레이션·표본추출)에만 영향을 줍니다. 데이터를 파일에서 그대로 읽어오는 것은 무작위가 아닙니다.

## 1. 필수 과제 — 로드 → EDA → 추론 → 회귀 → 예측 (한 흐름)

아래 1.1~1.10을 순서대로 완성합니다. 앞 단계의 결과(변수)가 뒤 단계로 이어지므로 **위에서부터 차례로** 실행하세요. 강의에서 Framingham으로 본 개념을 이번엔 여러분 손으로 재현합니다.

### 1.1 데이터 만나기 — 구조와 결측

이번 과제의 데이터는 **Framingham 심장연구**(framingham.csv, $n=4{,}240$)입니다. 미국 매사추세츠주 Framingham 주민들을 장기 추적한 역학 코호트로, 심혈관 위험요인 연구의 고전이에요. 강의에서 EDA→추론·검정→회귀를 관통한 바로 그 데이터를 과제에서도 그대로 씁니다.

**두 주인공을 먼저 소개할게요.**
- **`TenYearCHD`** — 이번 데이터의 **반응변수(response, 예측 대상)**. 0/1 이진값으로, 추적 10년 안에 **관상동맥심질환(CHD)** 이 발생했는지를 뜻합니다. 1이 발병(양성), 0이 무발병.
- **`sysBP`** — 이번 과제 내내 따라다닐 **주인공 연속변수**, 수축기 혈압(systolic blood pressure, mmHg). 심장이 수축할 때의 혈압이고, 값이 클수록 대체로 위험이 커진다고 알려져 있죠. 뒤 섹션의 CLT·부트스트랩·t검정·회귀가 전부 이 `sysBP`를 중심으로 돌아갑니다.

**결측(missing value)** 은 이번 데이터의 핵심 학습 포인트예요. 우리가 강의 앞에서 쓰던 tips 데이터는 결측이 하나도 없어서 그냥 불러 쓰면 됐지만, 실제 임상·역학 데이터는 거의 항상 값이 비어 있습니다(검사를 안 했거나, 응답을 안 했거나). Framingham도 마찬가지라, 예를 들어 **혈당(`glucose`)은 388명분이 비어 있어요.**

> **결측(NA/NaN)이란?** 그 칸의 값이 "0"이거나 "없음"이 아니라, **애초에 관측되지 않아 알 수 없는** 상태를 말합니다. pandas는 이를 `NaN`으로 표시하고, `df.isna().sum()`으로 열마다 몇 개가 비었는지 셀 수 있어요.

왜 지금부터 신경 쓰냐면, **결측을 어떻게 다루느냐에 따라 평균·신뢰구간·검정·회귀 결과가 달라지기** 때문입니다. 이번 섹션에서는 "어디에 결측이 얼마나 있는지"를 먼저 눈으로 확인하고, 실제 처리(해당 열 `dropna` 등)는 각 분석 섹션에서 그 분석에 필요한 열에 대해서만 하게 됩니다. 지금은 **데이터의 전체 지형을 파악하는 정찰 단계**라고 생각하세요.

In [ ]:
# 1) 데이터 크기: (행 개수, 열 개수)
print("shape (행, 열):", ____)        # TODO: df의 크기(행,열)를 출력

# 2) 앞부분 5행 미리보기 — 값이 어떻게 생겼는지 눈으로 확인
print("\n=== df.head() ===")
print(____)                            # TODO: 앞 5행 미리보기

# 3) 열별 자료형 + 비결측(non-null) 개수 한눈에 보기
print("\n=== df.info() ===")
____                                   # TODO: 열별 dtype/비결측 개수 요약

# 4) 열별 결측치 개수 (많은 순으로 정렬)
print("\n=== 결측치 개수 df.isna().sum() ===")
na_counts = ____.sort_values(ascending=False)   # TODO: 열별 결측 개수 세기
print(na_counts)

# 5) 결측이 있는 열만 골라 개수 + 비율(%)로 정리
na_only = na_counts[na_counts > 0]
na_table = pd.DataFrame({
    "결측개수": na_only,
    "결측비율(%)": (na_only / len(df) * 100).round(2),
})
print("\n=== 결측이 있는 열 (개수 + 비율) ===")
print(na_table)

# 6) 반응변수 TenYearCHD(0/1) 분포 확인
print("\n=== 반응변수 TenYearCHD 분포 ===")
print(df["TenYearCHD"].____)           # TODO: 0/1 각각 몇 개인지 세기
print("양성(CHD=1) 비율: {:.1%}".format(df["TenYearCHD"].mean()))

# 7) 주인공 연속변수 sysBP 요약 통계
print("\n=== 주인공 변수 sysBP 요약 ===")
print(df["sysBP"].____)                # TODO: 평균/표준편차/사분위 요약


**해석 질문 (본인이 방금 출력한 실제 값을 인용해서 답하세요.)**

1. `df.shape` 결과, 이 데이터의 **행(사람 수)** 과 **열(변수 수)** 은 각각 몇 개인가요? 그중 우리가 예측하려는 반응변수와 주인공 연속변수의 이름은 무엇인가요?

2. `df.isna().sum()` 에서 **결측이 가장 많은 열**은 무엇이고, 몇 개(전체의 몇 %)가 비어 있나요? 반대로 `sysBP`에는 결측이 몇 개 있나요?
   - 오개념 주의: 결측 개수가 `0`이라고 해서 "이 열은 완벽하다"는 뜻은 아니고, 단지 **관측되지 않은 칸이 없다**는 뜻일 뿐입니다. 또 `glucose`의 388개는 "혈당이 0"이 아니라 **값을 모른다**는 뜻이에요 — 0으로 채워 넣으면 평균이 왜곡됩니다.

3. `TenYearCHD.value_counts()` 를 보면 발병(1)과 무발병(0)이 각각 몇 명인가요? 양성 비율은 약 몇 %인가요? 두 집단 크기가 많이 차이 나는데(불균형), 이런 데이터에서 "무조건 0(무발병)이라고 예측"하면 정확도가 몇 %쯤 나올지 한 줄로 생각해 보세요.

4. tips 데이터와 달리 Framingham에는 결측이 있습니다. 만약 결측이 하나라도 있는 행을 **전부 삭제**한다면(모든 열 기준 `dropna`) 표본 크기가 크게 줄어들 수 있습니다. 이럴 때 "전체를 한 번에 dropna" 대신 **분석에 필요한 열만 골라 dropna** 하는 편이 왜 더 나을지, `glucose`의 결측 비율(약 9%)을 근거로 설명해 보세요. (뒤 섹션들이 실제로 이 방식을 씁니다.)

### 1.2 EDA · 상관관계 — 변수들이 서로 어떻게 얽혀 있나

앞 단계에서 각 변수를 하나씩(한 변수 분포) 봤다면, 이제는 **두 변수가 함께 움직이는 정도**를 봅니다. 이게 회귀로 넘어가기 전 꼭 필요한 준비 운동입니다.

**상관계수(correlation coefficient)**란: 두 연속형 변수가 같이 커지고 같이 작아지는 경향을 $-1 \sim +1$ 사이 한 숫자로 요약한 값입니다(여기선 가장 흔한 **Pearson** 상관). +1에 가까우면 "한쪽이 크면 다른 쪽도 크다"(양의 직선관계), −1에 가까우면 반대, 0에 가까우면 직선적 관계가 거의 없다는 뜻입니다.

- ⚠️ **오개념 주의 1 — 상관 ≠ 인과.** age와 sysBP(수축기 혈압)의 상관이 높다고 "나이가 혈압을 올린다"가 증명되는 건 아닙니다. 상관은 "같이 움직인다"만 말합니다.
- ⚠️ **오개념 주의 2 — 상관 0.05 ≈ 무관계가 아닙니다.** Pearson 상관은 **직선** 관계만 잡습니다. U자처럼 휘어진 강한 관계도 상관은 0이 나올 수 있습니다. 그래서 숫자(상관행렬)와 그림(산점도)을 **같이** 봅니다.

**상관행렬 히트맵**은 여러 변수의 쌍별 상관을 색으로 칠한 표입니다. 한눈에 "어떤 변수끼리 강하게 얽혔나"를 보여줍니다. 특히 우리가 눈여겨볼 것은 **sysBP와 diaBP(이완기 혈압)의 상관이 매우 높다($\approx 0.78$)**는 점입니다. 둘 다 "혈압"이라 거의 같은 정보를 담고 있어서, 나중에 **다중회귀에 둘을 동시에 넣으면 공선성(multicollinearity)** — 설명변수끼리 너무 닮아 각자의 계수 추정이 불안정해지는 문제 — 이 생깁니다. 이 히트맵이 그 문제를 **미리 예고**해 줍니다.

**산점도(scatter plot)**는 점 하나가 사람 한 명인 그림입니다. 히트맵의 한 숫자가 "실제로 어떤 모양의 관계"인지 눈으로 확인하는 용도입니다. age vs sysBP(뒤에서 단순회귀로 쓸 관계)와 sysBP vs diaBP(공선성의 실제 모양)를 그려 봅니다.

In [ ]:
try:
    num_cols
except NameError:
    num_cols = ["age", "sysBP", "diaBP", "totChol", "BMI", "glucose", "heartRate"]

# 1) 상관행렬 계산 (연속형 변수만). Pearson 상관.
#    힌트: DataFrame에는 상관행렬을 바로 구하는 메서드가 있다.
corr = df[num_cols].____
print("상관행렬 (Pearson):")
print(corr.round(3))

# 2) 히트맵으로 시각화
plt.figure(figsize=(8, 6))
sns.heatmap(
    ____,                       # TODO: 무엇을 히트맵으로 그릴까?
    annot=True, fmt=".2f",
    cmap="coolwarm", center=0,
    vmin=-1, vmax=1, square=True,
    cbar_kws={"label": "Pearson r"},
)
plt.title("Correlation matrix (Framingham num_cols)")
plt.tight_layout()
plt.show()

# 3) 관심 상관 두 개를 명시적으로 추출 (해석 질문에서 인용할 값)
#    힌트: corr는 행/열이 변수명인 표 → .loc[행, 열] 로 특정 칸을 꺼낸다.
corr_age_sysBP = corr.loc[____, ____]
corr_sysBP_diaBP = corr.loc[____, ____]
print(f"\nage - sysBP 상관계수 : {corr_age_sysBP:.3f}")
print(f"sysBP - diaBP 상관계수: {corr_sysBP_diaBP:.3f}")

# 4) 산점도 두 장: (a) age vs sysBP, (b) sysBP vs diaBP
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(data=df, x=____, y=____, alpha=0.3, s=15, ax=axes[0])  # TODO: age vs sysBP
axes[0].set_title(f"age vs sysBP  (r = {corr_age_sysBP:.3f})")
axes[0].set_xlabel("age")
axes[0].set_ylabel("sysBP")

sns.scatterplot(data=df, x=____, y=____, alpha=0.3, s=15, ax=axes[1])  # TODO: sysBP vs diaBP
axes[1].set_title(f"sysBP vs diaBP  (r = {corr_sysBP_diaBP:.3f})")
axes[1].set_xlabel("sysBP")
axes[1].set_ylabel("diaBP")

plt.tight_layout()
plt.show()


**해석 질문 (본인이 출력한 실제 값을 인용해 답하세요)**

1. **age–sysBP 상관계수를 소수 셋째 자리까지** 적어 보세요. 부호(+/−)와 크기를 근거로, 나이와 수축기 혈압의 관계를 한 문장으로 설명해 보세요. (예: "내 값은 0.___ 이고, 양수이므로 나이가 많을수록 sysBP가 대체로 …")

2. **sysBP–diaBP 상관계수**는 얼마였나요? 이 값이 age–sysBP 상관보다 훨씬 큰데, 왜 그럴까요? (힌트: 둘 다 무엇을 재는 값인가요?) 이렇게 서로 매우 닮은 두 변수를 **나중에 다중회귀에 함께** 넣으면 어떤 문제(공선성)가 생길지, 히트맵이 이를 어떻게 "예고"하고 있는지 본인 값을 들어 서술하세요.

3. ⚠️ 오개념 점검: 만약 어떤 두 변수의 상관이 0.02로 아주 작게 나왔다면, "두 변수는 아무 관계가 없다"고 결론지어도 될까요? 산점도를 함께 봐야 하는 이유와 연결해 답해 보세요.

### 1.3 표본분포와 중심극한정리(CLT) 시뮬레이션 — 손으로 직접 짜기

앞 절에서 본 `sysBP`(수축기 혈압) 전체 값을 이번엔 **'모집단(population)'** 이라고 상상해 봅시다. 현실에서는 모집단 전체를 다 볼 수 없고, 매번 **일부만 표본으로 뽑아** 평균을 냅니다. 그런데 "표본을 뽑아 평균을 내는 일"을 수천 번 반복하면, 그 **표본평균들이 그리는 분포**(= 표본분포, sampling distribution)에는 놀라운 규칙이 나타납니다.

- **표본분포(sampling distribution):** 같은 크기 `n`의 표본을 반복해서 뽑아 매번 평균을 냈을 때, 그 **평균값들**이 이루는 분포. 데이터 하나하나의 분포가 아니라 "평균의 분포"라는 점이 핵심입니다.
- **중심극한정리(CLT):** 모집단이 정규분포가 아니어도, 표본크기 `n`이 커지면 표본평균의 분포는 **정규분포에 가까워지고**, 그 **표준편차(= 표준오차, standard error)는 $\sigma/\sqrt{n}$** 으로 줄어듭니다. 여기서 $\sigma$는 모집단 표준편차입니다. `sysBP`는 $\sigma \approx 22$ 정도입니다.
- 직관: `n`이 커질수록 극단값들이 서로 상쇄돼 평균이 **덜 흔들립니다.** 그래서 히스토그램이 더 좁고 종 모양에 가까워집니다.

`np.random.choice` 같은 표본추출 함수는 써도 좋지만, **"반복해서 평균을 모으는 루프"는 라이브러리에 통째로 맡기지 말고 직접 `for` 루프로 구현**하세요. GPT에게 "CLT 시뮬레이션 코드 짜줘"라고 통째로 시키면 개념이 손에 남지 않습니다. 루프 한 줄씩 직접 쓰면서 "표본을 뽑는다 → 평균을 낸다 → 리스트에 담는다 → reps번 반복"의 흐름을 몸에 익히는 게 목표입니다.

In [ ]:
# --- 0) 재현성: 시드 고정 ---
np.random.seed(MY_SEED)

# --- 1) sysBP 전체 값을 '모집단'으로 사용 (결측 제거) ---
pop = df["sysBP"].____.values          # TODO: 결측 제거 메서드
sigma = pop.std(ddof=0)                 # 모집단 표준편차 ~ 22
print(f"모집단 크기 = {len(pop)},  모집단 평균 = {pop.mean():.2f},  모집단 표준편차(sigma) = {sigma:.2f}")

# --- 2) 표본평균을 모으는 헬퍼: 직접 for 루프로 구현 (라이브러리에 통째로 맡기지 말 것!) ---
def sample_means(data, n, reps):
    means = []
    for _ in range(reps):
        sample = ____                   # TODO: data에서 크기 n 표본을 복원추출 (np.random.choice, replace=True)
        means.append(____)              # TODO: 그 표본의 평균을 기록
    return np.array(means)

# --- 3) n=2 vs n=30 으로 표본분포 만들기 ---
reps = 5000
means_n2  = sample_means(pop, n=2,  reps=reps)
means_n30 = sample_means(pop, n=30, reps=reps)

# --- 4) 두 히스토그램 나란히 비교 ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

axes[0].hist(means_n2, bins=40, color="tab:orange", edgecolor="white")
axes[0].axvline(pop.mean(), color="black", linestyle="--", label="모집단 평균")
axes[0].set_title(f"표본평균 분포 (n=2, reps={reps})")
axes[0].set_xlabel("표본평균 sysBP"); axes[0].set_ylabel("빈도"); axes[0].legend()

axes[1].hist(means_n30, bins=40, color="tab:blue", edgecolor="white")
axes[1].axvline(pop.mean(), color="black", linestyle="--", label="모집단 평균")
axes[1].set_title(f"표본평균 분포 (n=30, reps={reps})")
axes[1].set_xlabel("표본평균 sysBP"); axes[1].legend()

plt.tight_layout()
plt.show()

# --- 5) 경험 표준편차(표준오차) vs 이론값 sigma/sqrt(n) 비교 ---
for n, means in [(2, means_n2), (30, means_n30)]:
    emp = means.std(ddof=0)             # 시뮬레이션으로 얻은 표본평균들의 표준편차
    theo = ____                         # TODO: CLT 이론값 = sigma / sqrt(n)
    print(f"n={n:>2} | 경험 표준편차 = {emp:.3f} | 이론값 sigma/sqrt(n) = {theo:.3f}")


**해석 질문 (본인이 출력한 실제 숫자를 꼭 인용해서 답하세요):**

1. `n=2`와 `n=30`의 히스토그램을 비교하세요. 어느 쪽이 더 좁고(폭이 작고) 종 모양에 가까웠나요? 두 분포의 중심은 모집단 평균(약 132)과 견주어 어디쯤에 있었나요?

2. 5번 셀에서 출력된 **경험 표준편차** 실제값을 두 개 다 적고, 각각을 **이론값 $\sigma/\sqrt{n}$** ($\sigma \approx 22$이므로 $22/\sqrt{2} \approx 15.6$, $22/\sqrt{30} \approx 4.0$)과 비교하세요. 얼마나 가깝게 맞았나요? `n`이 2에서 30으로 커질 때 표본평균의 흔들림(표준오차)은 어떻게 변했나요?

3. 위 결과를 **중심극한정리(CLT)** 한 문장으로 요약해 보세요. (힌트: "표본크기 n이 커지면 표본평균의 분포는 ___에 가까워지고, 그 표준편차는 ___으로 줄어든다.")

> ⚠️ 
> - **"표준편차가 줄어든다 = 데이터 자체가 덜 퍼진다"가 아닙니다.** 모집단(개별 sysBP 값)의 퍼짐 $\sigma \approx 22$는 그대로입니다. 줄어드는 건 **표본평균**의 퍼짐(=표준오차 $\sigma/\sqrt{n}$)입니다. `n`을 키운다고 혈압 데이터가 균일해지는 게 아니라, "평균을 추정하는 정밀도"가 좋아지는 것.
> - **CLT는 표본크기 n에 대한 규칙이지, 반복횟수 reps에 대한 규칙이 아닙니다.** reps를 5000→50000으로 늘리면 히스토그램이 더 매끄러워질 뿐, 분포의 폭(표준오차)은 `n`이 결정합니다. 폭을 줄이려면 `reps`가 아니라 `n`을 키워야 합니다.

### 1.4 신뢰구간 — 부트스트랩으로 sysBP 평균의 95% CI 직접 만들기

신뢰구간(confidence interval, CI)은 "표본으로 추정한 값이 얼마나 흔들릴 수 있는가"를 구간으로 표현한 것입니다. 우리는 전체 sysBP의 평균을 하나의 숫자(점추정)로 봤지만, 표본이 바뀌면 그 평균도 조금씩 달라집니다. 그 흔들림의 폭을 잡아주는 것이 CI입니다.

**부트스트랩(bootstrap)** 은 그 흔들림을 공식 없이 **재표집(resampling)** 으로 흉내 내는 방법입니다. 아이디어는 단순합니다.

1. 가진 데이터(pop, 크기 n)에서 **복원추출(replacement=중복 허용)** 로 다시 n개를 뽑습니다 → 가상의 새 표본 1개
2. 그 표본의 **평균**을 기록합니다
3. 1~2를 reps번(예: 2000번) 반복 → 평균이 reps개 쌓입니다(boot_means)
4. 이 평균들의 분포에서 **2.5 백분위 ~ 97.5 백분위**를 잘라내면 그 사이가 95% CI

여기서 핵심은 "복원추출"입니다. 같은 값을 여러 번 뽑을 수 있어야 매번 조금씩 다른 표본이 되고, 그 다름이 곧 평균의 흔들림이 됩니다. 이번 유형 A에서는 이 재표집 루프를 **손으로 직접** 짭니다 — `scipy.stats.bootstrap` 에 통째로 맡기지 않습니다. 루프를 직접 돌려봐야 "CI가 어디서 나오는 숫자인지"가 몸에 남습니다.

한 줄 풀이 — **백분위(percentile)**: 값들을 크기순으로 늘어놓았을 때 아래에서 몇 %지점에 오는 값. 2.5 백분위 = 아래쪽 2.5%를 잘라낸 자리, 97.5 백분위 = 위쪽 2.5%를 잘라낸 자리. 그 사이에 가운데 95%가 들어갑니다.

In [ ]:
# === 1.4 신뢰구간: sysBP 평균의 95% CI를 부트스트랩으로 직접 구현 ===
# pop 은 1.3에서 만든 전체 sysBP 값 배열입니다. (없다면 아래 한 줄로 복구)
pop = df["sysBP"].dropna().values

n = len(pop)          # 표본 크기 (매 재표집도 이 크기로 뽑는다)
reps = 2000           # 재표집 반복 횟수
rng = np.random.default_rng(MY_SEED)   # 재현 가능한 난수 생성기

# 부트스트랩 평균들을 담을 그릇
boot_means = np.empty(reps)

for i in range(reps):
    # TODO: pop 에서 복원추출(중복 허용)로 n개 인덱스를 뽑아라 (힌트: rng.integers)
    idx = ____
    resample = pop[idx]
    # TODO: 그 표본의 평균을 boot_means[i] 에 기록하라
    boot_means[i] = ____

# TODO: boot_means 에서 2.5 / 97.5 백분위를 잘라 ci_low, ci_high 로 받아라
ci_low, ci_high = ____

print(f"관측된 sysBP 평균 (점추정): {pop.mean():.2f}")
print(f"부트스트랩 95% 신뢰구간: [{ci_low:.2f}, {ci_high:.2f}]")

# 부트스트랩 평균들의 분포 + CI 시각화 (보일러플레이트 — 그대로 두세요)
plt.figure(figsize=(8, 4))
plt.hist(boot_means, bins=40, color="steelblue", edgecolor="white", alpha=0.8)
plt.axvline(pop.mean(), color="black", linestyle="-", linewidth=2, label=f"평균 {pop.mean():.2f}")
plt.axvline(ci_low,  color="crimson", linestyle="--", linewidth=1.5, label=f"2.5% {ci_low:.2f}")
plt.axvline(ci_high, color="crimson", linestyle="--", linewidth=1.5, label=f"97.5% {ci_high:.2f}")
plt.title("Bootstrap distribution of sysBP mean (reps=2000)")
plt.xlabel("resampled mean of sysBP")
plt.ylabel("frequency")
plt.legend()
plt.tight_layout()
plt.show()


**해석 질문 (본인이 출력한 실제 숫자를 인용해 답하세요)**

1. 방금 얻은 신뢰구간 `[ci_low, ci_high]`의 실제 값을 적으세요. 강의에서 예고한 대략 [131.7, 133.0]과 비슷한가요? (완전히 같지 않아도 됩니다 — `MY_SEED`와 reps에 따라 소수점이 달라집니다.)

2. **빈도주의 해석을 정확히 쓰세요.** 다음 두 문장 중 옳은 것은 무엇이고, 왜 나머지는 틀렸는지 본인 CI 값을 넣어 설명하세요.
   - (A) "이런 방식으로 표본을 뽑아 CI를 만드는 절차를 무한히 반복하면, 그렇게 만들어진 구간들 중 약 95%가 진짜 모평균을 포함한다."
   - (B) "진짜 sysBP 모평균이 내 구간 [본인 값] 안에 있을 확률이 95%다."

   → 정답은 (A)입니다. **오개념 주의**: 빈도주의에서 모평균은 고정된(움직이지 않는) 값이고, 무작위로 흔들리는 것은 우리가 만든 **구간** 쪽입니다. 그래서 확률 95%는 "이 특정 구간"이 아니라 "구간을 만드는 절차 전체"에 붙는 성질입니다. 이미 계산이 끝난 내 구간 하나는 참값을 "포함했거나 / 안 했거나" 둘 중 하나일 뿐, 여기에 95% 확률을 매기는 (B)는 베이지안식 표현이라 이 프레임에서는 부정확합니다.

3. reps를 200으로 줄이면 CI 폭과 히스토그램 모양이 어떻게 달라질지 예상하고, 실제로 바꿔 실행해 확인해 보세요. (재표집 횟수는 CI의 '안정성'에 영향을 주지, 데이터 자체의 불확실성을 줄여주지는 않는다는 점에 주목.)

### 1.5 두 집단 평균차 t검정 (유형 B)

이제 "CHD가 생긴 사람과 안 생긴 사람은 수축기 혈압(sysBP)이 정말 다른가?"를 통계적으로 검정합니다. 눈으로 평균 두 개를 비교해 "달라 보인다"고 말하는 건 근거가 약합니다. **두 집단 평균차 t검정(two-sample t-test)** 은 "두 집단의 평균이 같다(차이가 0이다)"는 **귀무가설 $H_0$** 을, 관측된 평균차가 표본추출의 우연만으로 나올 확률(**p값**)로 판단합니다.

**왜 Welch t인가요?** 기본형(Student t)은 두 집단의 **분산이 같다**고 가정합니다. 하지만 여기서는 두 군의 sysBP 표준편차가 다릅니다(CHD군 $\approx 26.7$ vs 비CHD군 $\approx 20.5$ — 고혈압인 사람들이 모인 CHD군이 더 넓게 퍼짐). 분산이 다를 때는 **Welch t검정**(`equal_var=False`)이 자유도를 보정해 더 정확합니다. 실무에서는 "분산이 같은지 애매하면 그냥 Welch"가 안전한 기본값입니다.

**p값 한 줄 풀이:** p값은 "$H_0$(차이 없음)이 참이라고 가정했을 때, 지금 본 것만큼 또는 그보다 더 극단적인 평균차가 우연히 관측될 확률"입니다. 작을수록(관례상 < 0.05) "우연으로 보기엔 너무 극단적 → $H_0$ 기각"이 됩니다.

**주의 — 유의성 ≠ 크기.** p값은 "차이가 0이 아니다"만 말해줄 뿐, **차이가 얼마나 큰지**는 말해주지 않습니다. n이 크면 아주 작은 차이도 p가 극단적으로 작아집니다. 그래서 p값과 함께 **효과크기(평균차 자체)** 를 반드시 같이 봐야 합니다.

In [ ]:
# 1.5 두 집단 평균차 t검정 (Welch) ----------------------------------
# TenYearCHD == 1 (10년 내 CHD 발생) vs == 0 각각의 sysBP (결측 제거)
g1 = df.loc[df["TenYearCHD"] == 1, "sysBP"].dropna().values   # CHD 발생군
g0 = df.loc[____, "sysBP"].dropna().values                    # TODO: 비발생군 조건

print(f"CHD군    (n={len(g1)}): 평균 {g1.mean():.1f}, 표준편차 {g1.std(ddof=1):.1f}")
print(f"비CHD군  (n={len(g0)}): 평균 {g0.mean():.1f}, 표준편차 {g0.std(ddof=1):.1f}")

# 두 군의 분산이 다르므로(26.7 vs 20.5) Welch t검정
# TODO: stats.ttest_ind 로 Welch t검정 (분산 다름 옵션을 꼭 넣을 것)
t_stat, p_val = ____

mean_diff = g1.mean() - g0.mean()   # 효과크기: 평균차
print(f"\n평균차(효과크기) = {mean_diff:.1f} mmHg")
print(f"Welch t = {t_stat:.2f},  p = {p_val:.2e}")

if p_val < 0.05:
    print("=> p < 0.05: 귀무가설(두 군 평균 같다) 기각")
else:
    print("=> p >= 0.05: 귀무가설 기각 실패")


**해석 질문 (본인이 출력한 실제 값을 인용해 답하세요)**

1. 당신이 계산한 `t_stat`과 `p_val`은 각각 얼마인가요? 이 p값을 근거로 "두 군의 sysBP 평균은 같다"는 귀무가설을 **기각하나요, 기각에 실패하나요?** 한 문장으로 판정하세요.

2. p값을 "CHD 발생군과 비발생군의 sysBP가 우연히 이만큼 차이 날 확률"이라고 말하면 왜 부정확한가요? p값의 정의(귀무가설이 참일 때…)를 살려 한 줄로 고쳐 쓰세요.
   - (오개념 주의) p값은 "귀무가설이 참일 확률"이 **아닙니다.** "귀무가설이 참이라고 **가정했을 때** 이만큼 극단적인 결과가 나올 확률"입니다.

3. p값이 아주 작다는 것만으로 "혈압 차이가 임상적으로 크다"고 결론 내릴 수 있나요? 당신이 출력한 **평균차($\approx 13.3$ mmHg)** 를 함께 인용해, p값(유의성)과 효과크기(차이의 크기)를 **왜 둘 다** 봐야 하는지 설명하세요.
   - (힌트) 표본 n이 매우 크면 아주 작은 평균차도 p가 극단적으로 작아집니다.

4. 이 검정에서 `equal_var=False`(Welch)를 쓴 이유를, 두 군의 **표준편차($\approx 26.7$ vs $\approx 20.5$)** 를 인용해 한 문장으로 답하세요. 만약 `equal_var=True`로 바꾸면 무엇이 달라질지 예상해 보세요.

### 1.6 단순선형회귀 (Simple Linear Regression)

이제 두 변수의 관계를 **직선 하나로 요약**해 봅니다. 1.5에서 "CHD군이 sysBP가 높다"처럼 *집단 비교*를 했다면, 회귀는 한 걸음 더 나아가 **"나이가 1살 늘면 수축기혈압(sysBP)은 평균 몇 mmHg 변하는가?"** 라는 *연속적인 관계의 크기*를 숫자로 답합니다.

- **단순선형회귀란?** 설명변수(x, 여기선 `age`) 하나로 반응변수(y, `sysBP`)를 $y \approx \beta_0 + \beta_1 \cdot x$ 형태의 직선으로 예측하는 모형입니다. $\beta_1$(기울기, slope)이 바로 "x 1단위 증가 시 y의 평균 변화량"입니다.
- **OLS(Ordinary Least Squares, 최소제곱법):** 실제 값과 직선 예측값의 차이(잔차)를 제곱해 합한 값을 **가장 작게** 만드는 $\beta_0, \beta_1$을 찾는 방법입니다. statsmodels의 `smf.ols`가 이 계산을 해 줍니다.
- **`smf.ols("sysBP ~ age", ...)`:** `~` 왼쪽이 반응변수 y, 오른쪽이 설명변수 x인 **R 스타일 공식(formula)** 문법입니다. `.fit()`을 붙여야 실제로 계수가 추정됩니다.
- **$R^2$(결정계수):** 모형이 y의 전체 변동(분산) 중 몇 %를 설명하는지를 0~1로 나타냅니다. $R^2=0.155$면 sysBP 변동의 약 15.5%를 나이만으로 설명한다는 뜻입니다. **`age`와 `sysBP`의 상관계수($\approx 0.394$)를 제곱하면 정확히 $R^2$가 됩니다** — 단순회귀에서 $R^2 = r^2$.
- **계수의 p값·95% 신뢰구간:** 기울기 $\beta_1$에도 1.5의 t검정과 같은 논리가 적용됩니다. "진짜 기울기가 0(=관계 없음)이다"라는 귀무가설을 검정하고, $\beta_1$이 있을 법한 범위를 95% CI로 줍니다. CI가 0을 포함하지 않으면 통계적으로 유의합니다.

> 참고: `age`와 `sysBP`는 결측이 없는 열이라 이 절에서는 별도 dropna가 필요 없습니다. (결측이 있는 열로 회귀할 때는 해당 열들을 함께 dropna 해야 합니다.)


In [ ]:
# age(설명변수) 하나로 sysBP(반응변수)를 예측하는 단순선형회귀
# TODO: "반응변수 ~ 설명변수" 공식으로 OLS 모형을 적합하세요 (.fit()까지)
model_simple = smf.ols(____, data=df).fit()

# 회귀 결과 전체 요약표: 계수, 표준오차, t, p값, 95% CI, R^2 등
print(model_simple.summary())

# 핵심 값만 뽑아 보기
beta_age = model_simple.params["age"]          # age 기울기 계수
p_age    = model_simple.pvalues["age"]         # age 계수의 p값
ci_age   = model_simple.conf_int().loc["age"]  # age 계수의 95% 신뢰구간 [하한, 상한]
# TODO: 모형의 결정계수(R^2)를 꺼내세요 (model_simple 의 속성)
r2       = ____

print(f"\nage 계수(기울기)  = {beta_age:.4f} mmHg / 1살")
print(f"age 계수 p값       = {p_age:.3e}")
print(f"age 계수 95% CI    = [{ci_age[0]:.4f}, {ci_age[1]:.4f}]")
print(f"R^2               = {r2:.4f}")

# 단순회귀에서 R^2 = (상관계수)^2 임을 직접 확인
r = df["age"].corr(df["sysBP"])
print(f"\ncorr(age, sysBP)  = {r:.4f},  r^2 = {r**2:.4f}  (== R^2)")

# 산점도 + 회귀직선으로 관계 시각화
plt.figure(figsize=(6, 4))
sns.regplot(x="age", y="sysBP", data=df,
            scatter_kws={"alpha": 0.15, "s": 10}, line_kws={"color": "red"})
plt.title("sysBP ~ age (simple linear regression)")
plt.xlabel("age (years)")
plt.ylabel("sysBP (mmHg)")
plt.tight_layout()
plt.show()


**해석 질문 (본인이 출력한 실제 값을 인용해 답하세요):**

1. 본인이 얻은 `age` 계수는 얼마입니까? 이 값을 문장으로 옮겨 보세요: "나이가 1살 많아질수록 수축기혈압(sysBP)은 평균 약 ____ mmHg 높아진다." (숫자를 직접 채워 쓰기)

2. `age` 계수의 p값과 95% 신뢰구간을 인용하세요. 이 CI가 **0을 포함하는지** 확인하고, 그 사실이 "나이와 혈압 사이에 관계가 있다"는 주장과 어떻게 연결되는지 한 문장으로 설명하세요.

3. 본인의 $R^2$는 얼마입니까? 그리고 `corr(age, sysBP)`의 제곱과 같은 값이 나왔나요? "$R^2$ = ____ 이므로, 나이만으로 sysBP 변동의 약 ____%를 설명한다"를 채워 쓰세요.

**오개념 주의:**
- ❌ "$R^2$가 0.155니까 모형이 나쁘다/틀렸다" → $R^2$가 낮다고 관계가 없는 게 아닙니다. 계수의 p값($\approx 10^{-157}$)은 관계가 **매우 확실**함을 보여줍니다. $R^2$는 "예측의 정확도(설명력)"이고 p값은 "관계의 존재 여부"로, 서로 다른 질문입니다.
- ❌ "나이가 혈압을 올린다(인과)" → 회귀 계수는 **연관(상관)**의 크기일 뿐, 이 관측 데이터만으로 인과를 증명하지 못합니다. "높아지는 경향이 있다" 정도로 표현하세요.
- ❌ 계수 단위를 빼먹기 → 기울기는 단순한 숫자가 아니라 **mmHg / 1살**이라는 단위를 가집니다.


### 1.7 잔차 진단 — 회귀 가정을 눈으로 검사하기

1.6에서 만든 단순회귀 `model_simple`(`sysBP ~ age`)은 "직선 하나"를 데이터에 맞췄습니다. 그런데 이 직선이 믿을 만한지는 $R^2$만으로는 알 수 없습니다. **선형회귀가 전제하는 가정들이 실제로 성립하는지**를 확인해야 하는데, 그 가장 좋은 도구가 바로 **잔차(residual)** 입니다.

**잔차** = 실제값 − 예측값. 즉 각 사람의 진짜 `sysBP`에서 모델이 예측한 값을 뺀 "빗나간 양"입니다. 모델이 잘 맞았다면 잔차는 특정 패턴 없이 0 주변에 무작위로 흩어져야 합니다.

이 섹션에서는 두 가지 그림을 그립니다.

### 잔차 vs 적합값(fitted) 산점도
가로축에 **적합값**(fitted = 모델이 예측한 sysBP), 세로축에 잔차를 찍습니다. 여기서 보고 싶은 것:
- **등분산성(homoscedasticity)**: 예측값이 크든 작든 잔차의 흩어짐 폭이 일정해야 합니다. 만약 오른쪽으로 갈수록 점들이 위아래로 넓게 퍼지는 **깔때기(fan/funnel) 모양**이 보이면 등분산 가정이 깨진 것입니다. `sysBP`는 오른쪽으로 긴 꼬리(우측치우침)를 가진 변수라 혈압이 높은 사람들에서 잔차가 더 크게 튈 가능성이 있습니다.
- **선형성**: 잔차가 0을 중심으로 위아래 균형을 이뤄야 합니다. 곡선 형태의 추세가 보이면 관계가 직선이 아니라는 신호입니다.

### Q-Q 플롯(정규 분위수-분위수 그림)
잔차가 **정규분포**를 따르는지 봅니다. 잔차를 크기순으로 세운 뒤, 정규분포에서 기대되는 값과 짝지어 찍습니다. 잔차가 정규라면 점들이 **대각 직선 위에 놓입니다**. 양 끝(꼬리)에서 점들이 직선을 벗어나 위로/아래로 휘면, 실제 분포가 정규보다 **꼬리가 두껍다**는 뜻입니다. 우측치우침 변수에서는 보통 **오른쪽 위 끝이 직선 위로 치솟는** 모양이 나옵니다.

한 줄 요약: 잔차 그림은 "직선을 믿어도 되는가"를 검사하는 건강검진표입니다. 깔때기나 꼬리 이탈이 보이면, 다음 단계로 **로그 변환**(`log(sysBP)`)이나 **robust 회귀** 같은 처방을 고려하게 됩니다 — 이건 뒤에서 다룹니다.

In [ ]:
# 1.6에서 적합한 model_simple(sysBP ~ age)에서 적합값과 잔차 추출
# 힌트: fittedvalues, resid 속성을 쓴다
fitted = ____              # TODO: model_simple의 적합값
resid = ____               # TODO: model_simple의 잔차

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# (1) 잔차 vs 적합값 산점도
axes[0].scatter(____, ____, s=10, alpha=0.3, edgecolor="none")   # TODO: (x=적합값, y=잔차)
axes[0].axhline(0, color="red", linestyle="--", linewidth=1)     # 기준선 y=0
axes[0].set_xlabel("Fitted values (predicted sysBP)")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals vs Fitted")

# (2) Q-Q 플롯 (잔차의 정규성)
# 힌트: stats.probplot(데이터, dist="norm", plot=축)
____                        # TODO: 잔차의 Q-Q 플롯을 axes[1]에 그리기
axes[1].set_title("Normal Q-Q plot of residuals")

plt.tight_layout()
plt.show()

# 진단용 요약 수치
print(f"잔차 평균: {resid.mean():.4f}  (0에 가까워야 정상)")
print(f"잔차 표준편차: {resid.std():.3f}")
print(f"잔차 왜도(skewness): {stats.skew(resid):.3f}  (0에서 멀수록 비대칭/치우침)")
print(f"잔차 첨도(kurtosis, 초과): {stats.kurtosis(resid):.3f}  (0보다 크면 정규보다 두꺼운 꼬리)")


**해석 질문 (본인이 출력한 그림과 수치를 직접 인용해서 답하세요)**

1. **잔차 vs 적합값** 산점도에서, 적합값이 커질수록(예측 혈압이 높아질수록) 잔차의 흩어짐 폭이 넓어지는 **깔때기(funnel) 모양**이 보이나요? 보인다면 이는 어떤 가정(등분산성)이 위반되었다는 뜻인가요? 안 보인다면 어떤 근거로 그렇게 판단했나요?

2. **Q-Q 플롯**에서 점들이 대각 직선을 잘 따라가나요, 아니면 **양 끝(꼬리)에서 벗어나**나요? 특히 오른쪽 위 끝이 직선 위로 치솟는지 확인하고, 본인이 출력한 **왜도(skewness) 값**을 인용해 잔차가 어느 쪽으로 치우쳐 있는지 설명하세요. (`sysBP`의 우측치우침과 연결지어 보세요.)

3. 위 두 진단을 종합할 때, 이 단순회귀 결과를 그대로 신뢰해도 될까요? 만약 등분산·정규성 가정이 흔들린다면, 다음 단계로 어떤 처방(예: `sysBP`에 **로그 변환**, 또는 이상치에 덜 민감한 **robust 회귀**)을 고려할 수 있을지 한 줄로 적어보세요.

> ⚠️ **오개념 주의**: 잔차 평균이 0에 가깝다고 해서(OLS에서는 항상 그렇습니다) "가정이 다 만족됐다"는 뜻은 아닙니다. 평균이 0인 것은 최소제곱법의 **수학적 결과**일 뿐이고, 우리가 검사하는 것은 잔차의 **퍼짐(등분산)과 모양(정규성)** 입니다. 또한 $R^2$가 낮다고 잔차 진단이 무의미한 것도 아닙니다 — 진단은 "관계의 강도"가 아니라 "모델 형태의 적절성"을 봅니다.

### 1.8 다중회귀 — 여러 변수로 sysBP 설명하기 (범주형 포함)

단순회귀(1.6)는 `age` 하나로 `sysBP`를 설명했습니다. 하지만 혈압은 나이만으로 정해지지 않습니다 — 체중(BMI), 콜레스테롤, 혈당, 성별, 흡연, 당뇨 등이 함께 작용합니다. **다중회귀(multiple regression)** 는 이 여러 예측변수를 한 모델에 넣어, "다른 변수를 고정했을 때 각 변수가 sysBP에 주는 순효과(partial effect)"를 추정합니다.

- **연속형 변수**(`age`, `BMI`, `totChol`, `glucose`)는 그대로 넣습니다.
- **범주형 변수**(`male`, `currentSmoker`, `diabetes`)는 0/1 코드지만, 숫자 크기에 의미가 없으므로 `C(...)`로 감싸 **더미(dummy) 변수**로 처리합니다. `C(male)`은 "여성(0) 대비 남성(1)의 sysBP 차이"를 계수로 줍니다.

**결측(NaN) 자동 제거 주의:** 이 모델에는 `glucose`(결측 388개)·`totChol`(50개)·`BMI`(19개) 등 결측 있는 열이 들어갑니다. statsmodels는 formula에 쓰인 열 중 **하나라도 NaN인 행을 통째로 제외**(listwise deletion)합니다. 그래서 전체 4,240행보다 **분석에 실제 쓰인 n이 줄어듭니다**(`model_multi.nobs`로 확인). 어떤 변수를 넣느냐에 따라 표본 크기가 달라진다는 점이 결측 데이터의 핵심 함정입니다.

**$R^2$는 단순회귀보다 오릅니다:** 설명변수를 늘리면 모델이 잡아내는 분산이 늘어 결정계수 $R^2$가 상승합니다. 다만 변수를 넣을수록 $R^2$는 기계적으로 커지는 성질이 있어, 뒤(1.9)에서 잔차·계수의 의미와 공선성을 함께 따져봐야 합니다.

In [ ]:
# 1.8 다중회귀 (범주형 포함)
# 연속형(age, BMI, totChol, glucose) + 범주형은 C(...)로 감싸 더미 처리
# statsmodels가 NaN 행을 자동 제외 → 실제 n이 줄어드는 점에 주의

# TODO: sysBP를 age, BMI, totChol, glucose(연속형) + male, currentSmoker, diabetes(범주형)로 설명하는 OLS
#       범주형 세 변수는 반드시 C(...)로 감쌀 것
model_multi = smf.ols(
    "____",
    data=df
).fit()

print(model_multi.summary())

# 결측 자동 제거로 실제 사용된 표본 크기 확인
print(f"\n전체 행수: {len(df)}")
print(f"다중회귀에 실제 사용된 n(nobs): {int(model_multi.nobs)}")   # TODO: model_multi에서 nobs 속성 사용
print(f"→ 결측으로 {len(df) - int(model_multi.nobs)}개 행 제외됨")

# 단순회귀(1.6) 대비 R² 상승 확인
print(f"\n단순회귀(age만) R²   = {model_simple.rsquared:.3f}")
print(f"다중회귀 R²          = {____:.3f}")   # TODO: model_multi의 R²
print(f"다중회귀 Adjusted R² = {model_multi.rsquared_adj:.3f}")


**Q. 다중회귀 결과를 확인하세요.**

1. `model_multi.nobs`(모형에 실제로 쓰인 관측 수)는 전체 4,240에서 얼마로 줄었나요? **본인 값**을 적고, 왜 줄었는지 한 줄로 설명하세요. (힌트: statsmodels는 수식에 들어간 변수 중 하나라도 결측인 행을 자동 제외합니다.)
2. `age`의 계수가 단순회귀(1.6)와 다중회귀에서 달라졌나요? 어느 쪽이 더 크고 왜 그럴 수 있을까요? (다른 변수를 '통제'한다는 게 무슨 뜻인지와 연결해 보세요.)

### 1.9 계수 유의성 · R²/Adjusted R² · 다중공선성(VIF)

앞에서 단순회귀(`model_simple`, `sysBP ~ age`)와 다중회귀(`model_multi`, `sysBP ~ age + BMI + totChol + glucose`)를 적합했습니다. 이제 모델을 "믿을 수 있는가"를 세 가지 눈으로 점검합니다.

### 계수 유의성 (p-value)
회귀계수 옆의 **p-value**는 "이 변수의 진짜 효과가 0(=관계 없음)인데, 우연히 이 정도 계수가 나올 확률"입니다. 관례상 **$p < .05$**면 "0이라고 보기 어렵다 = 통계적으로 유의하다"고 말합니다. 주의: p가 작다고 효과가 **큰** 것은 아닙니다. p는 "0이 아니라는 증거의 강도"일 뿐, 효과 크기는 계수값 자체가 말해줍니다.

### R² vs Adjusted R²
- **$R^2$(결정계수)**: 반응변수(sysBP)의 분산 중 모델이 설명한 비율(0~1). 예측변수를 **아무거나** 더 넣어도 $R^2$는 절대 줄지 않습니다 — 쓸모없는 변수도 $R^2$를 조금은 올립니다.
- **Adjusted $R^2$(수정 결정계수)**: 변수 개수에 벌점을 준 $R^2$. 변수를 추가했을 때 **설명력 증가가 벌점보다 작으면 오히려 떨어집니다.** 그래서 변수 수가 다른 모델(단순 vs 다중)을 비교할 땐 $R^2$가 아니라 **Adjusted $R^2$로 비교**하는 게 공정합니다.

### 다중공선성과 VIF
다중공선성(multicollinearity)은 예측변수들끼리 서로 강하게 상관돼 "누가 효과를 낸 건지" 회귀가 구분하지 못하는 상태입니다. 계수가 불안정해지고 표준오차가 커집니다(=p-value가 엉뚱하게 커짐).

이를 진단하는 지표가 **VIF(Variance Inflation Factor, 분산팽창인자)**입니다. 각 예측변수를 "나머지 예측변수들로 회귀"했을 때의 $R^2 (= R_j^2)$로 **$\mathrm{VIF}_j = 1 / (1 - R_j^2)$** 로 계산합니다. 그 변수가 다른 변수들로 얼마나 잘 예측되는지(=중복되는지)를 재는 것입니다.
- **$\mathrm{VIF} \approx 1$**: 다른 변수와 거의 독립 (건강)
- **VIF 5 이상**: 주의, **10 이상**: 심각한 공선성 (해당 계수 해석 신뢰 어려움)

여기서는 `[age, BMI, totChol, glucose]` 네 변수의 VIF를 계산합니다. 이 조합은 서로 별 상관이 없어 **VIF가 모두 1 근처** — 즉 "공선성 걱정 없는 깨끗한 예제"입니다. 그래서 이번 목표는 "문제를 잡는 것"이 아니라 **VIF를 읽는 법을 익히는 것**입니다.

대비로, 1.2/1.8에서 본 **sysBP–diaBP 상관 0.78**처럼 강하게 얽힌 두 변수를 **동시에** 예측변수로 넣으면 VIF가 곧장 올라갑니다(둘만이면 $\mathrm{VIF} \approx 1/(1 - 0.78^2) \approx 2.6$). 같은 정보를 두 번 넣는 셈이기 때문입니다. 상관이 더 세질수록(0.9, 0.95…) VIF는 10을 훌쩍 넘겨 폭증합니다. 이번 셀에서 그 방향을 직접 눈으로 확인합니다.

> 참고: `add_constant`로 붙인 상수항(const)의 VIF는 높게 나오는데, 이는 예측변수의 평균 위치 때문에 생기는 정상적 현상이라 **해석하지 않습니다.** VIF는 실제 예측변수들만 봅니다.

In [ ]:
# ── (1) 단순 vs 다중: R²와 Adjusted R² 비교 ─────────────────────────────
print("[단순회귀] sysBP ~ age")
print(f"  R²        = {model_simple.rsquared:.4f}")
# TODO: model_simple의 '수정' 결정계수 속성을 채워라 (____)
print(f"  Adj. R²   = {model_simple.____:.4f}")
print("\n[다중회귀] sysBP ~ age + BMI + totChol + glucose")
print(f"  R²        = {model_multi.rsquared:.4f}")
print(f"  Adj. R²   = {model_multi.____:.4f}")
print(f"\nAdjusted R² 변화 (단순→다중): "
      f"{model_multi.rsquared_adj - model_simple.rsquared_adj:+.4f}")

# ── (2) 다중회귀 계수 유의성: p < .05 인 변수 골라내기 ────────────────────
pvals = model_multi.pvalues.drop("Intercept")
print("\n[계수 p-value]")
print(pvals.to_string())

# TODO: p-value가 0.05 미만인 변수만 남기도록 조건을 채워라
sig_vars = pvals[____].index.tolist()
print(f"\n유의한(p<.05) 변수: {sig_vars}")

# ── (3) VIF 계산: 수치형 예측변수 [age, BMI, totChol, glucose] ────────────
vif_vars = ["age", "BMI", "totChol", "glucose"]
X_vif = df[vif_vars].dropna()          # VIF도 결측이 있으면 계산 불가 → dropna
X_vif = sm.add_constant(X_vif)         # variance_inflation_factor는 상수항 필요

vif_table = pd.DataFrame({
    "variable": X_vif.columns,
    # TODO: i번째 열의 VIF를 반환하는 함수를 호출하라 (X_vif.values, i 사용)
    "VIF": [____(X_vif.values, i)
            for i in range(X_vif.shape[1])]
})
print("\n[VIF] (const 행은 해석하지 않음)")
print(vif_table.to_string(index=False))

# ── (4) 대비: 강한 공선성(sysBP–diaBP, r≈0.78)을 일부러 넣어보기 ──────────
demo_vars = ["age", "BMI", "totChol", "glucose", "sysBP", "diaBP"]
X_demo = sm.add_constant(df[demo_vars].dropna())
vif_demo = pd.DataFrame({
    "variable": X_demo.columns,
    "VIF": [variance_inflation_factor(X_demo.values, i)
            for i in range(X_demo.shape[1])]
})
print(f"\n[대비용 VIF] sysBP–diaBP 상관 = {df['sysBP'].corr(df['diaBP']):.3f}")
print(vif_demo.to_string(index=False))
print("→ 서로 얽힌 sysBP·diaBP를 함께 넣으면 두 변수의 VIF만 1 → 3 부근으로 상승한다.")


**해석 질문 (본인이 출력한 실제 값을 인용해 답하세요 — 복붙 금지)**

1. **유의성:** 다중회귀에서 `p < .05`로 유의하게 나온 변수를 모두 나열하세요. 몇 개인가요? (본인 `sig_vars` 출력값 인용) — 여기서는 네 변수가 다 유의할 텐데, "p가 가장 작은 변수가 sysBP에 가장 큰 영향을 준다"고 말할 수 있나요? 없다면 왜인가요? (힌트: p-value와 계수 크기는 다른 것)

2. **VIF 읽기:** `vif_table`에서 `age·BMI·totChol·glucose`의 VIF 값을 적고, 이 값들이 걱정할 수준인지 판단하세요(기준: 5 이상 주의, 10 이상 심각). 본인 값 기준으로 "공선성 문제가 있다/없다"를 한 문장으로 결론지으세요. (const 행의 큰 VIF는 왜 무시하나요?)

3. **대비 확인:** `vif_demo`에서 `sysBP`와 `diaBP`의 VIF는 각각 얼마였나요? 두 변수의 상관이 0.78인데 VIF가 이렇게 변한 이유를 한 줄로 설명하세요.

4. **모델 비교:** 단순회귀와 다중회귀의 **Adjusted $R^2$**를 각각 적고, 변화량을 인용하세요. Adjusted $R^2$가 (줄지 않고) 올랐다는 사실은 무엇을 뜻하나요? — 그리고 단순 vs 다중 모델을 비교할 때 왜 그냥 $R^2$가 아니라 Adjusted $R^2$로 봐야 하는지 한 문장으로 쓰세요.

> **오개념 주의:** ① "$R^2$가 오르면 무조건 더 좋은 모델" ✗ — $R^2$는 변수를 넣으면 항상 오르므로, 변수 수가 다른 모델 비교엔 Adjusted $R^2$를 씁니다. ② "VIF가 낮으니 이 모델은 완벽합니다" ✗ — VIF는 공선성만 볼 뿐, 유의성·잔차 가정·설명력과는 별개의 점검 항목입니다.

### 1.10 해석(explain) vs 예측(predict) — train/test로 갈라 보기

지금까지 우리가 한 회귀(1.6~1.9, `smf.ols`)는 대부분 **해석(explanation)** 관점이었어요. "age가 1 늘면 sysBP가 몇 오르나?", "이 계수가 통계적으로 유의한가?", "모델이 데이터의 분산을 얼마나 설명하나($R^2$)?" 같은 질문이죠. 이때 $R^2$는 **모델을 학습시킨 바로 그 데이터(in-sample)** 위에서 계산됩니다.

그런데 현실에서 던지는 또 다른 질문이 있어요: **"새 환자가 오면 그 사람의 sysBP를 얼마나 잘 맞힐까?"** 이건 **예측(prediction)** 관점입니다. 예측 성능은 모델이 **한 번도 본 적 없는 데이터**에서 얼마나 틀리는지로 재야 공정해요. 학습에 쓴 데이터로 성능을 재면, 학생이 시험 문제를 미리 보고 푼 것과 같아서 성능이 부풀려집니다.

그래서 데이터를 둘로 쪼갭니다.
- **train(훈련) 세트**: 모델이 계수를 배우는 데 쓰는 데이터.
- **test(테스트) 세트**: 학습에는 절대 안 쓰고, 마지막에 "처음 보는 데이터"인 척하며 성능만 재는 데이터.

이번 섹션에서 쓸 예측 오차 지표 두 가지:
- **RMSE (Root Mean Squared Error, 평균제곱근오차)**: 오차를 제곱해 평균 낸 뒤 제곱근. 큰 오차에 더 큰 벌점을 줍니다(제곱 때문에). 단위는 예측 대상과 같아요(여기선 mmHg).
- **MAE (Mean Absolute Error, 평균절대오차)**: 오차의 절댓값 평균. 모든 오차를 똑같이 취급하고, 이상치(outlier)에 덜 민감합니다.

두 지표 모두 **작을수록 좋고**, 항상 $\mathrm{RMSE} \ge \mathrm{MAE}$ 입니다(제곱이 큰 오차를 키우니까). 둘의 **간격이 크면** = 유독 크게 빗나간 예측(큰 잔차)이 몇 개 있다는 신호예요.

한 가지 주의: 우리가 **예측하려는 값(sysBP)** 자체나, 그것과 사실상 같은 정보를 X(설명변수)에 넣으면 안 됩니다. 이걸 **데이터 누수(leakage)**라고 하고, test 성능을 비현실적으로 좋게 만들어요. 그래서 sysBP는 y로만 쓰고 X에서는 뺍니다.

이 "해석이냐 예측이냐"의 구분이 이번 주 리뷰 논문 Shmueli, "To Explain or to Predict?" (2010)의 핵심 주제입니다. 같은 회귀식이라도 **무엇을 묻느냐**에 따라 좋은 모델·좋은 지표가 달라진다는 이야기예요.

In [ ]:
# === 1.10 해석 vs 예측: train/test 분할로 예측 성능 재기 ===
# 관점 전환: 1.6~1.9의 statsmodels R²(in-sample, 해석) → 여기선 sklearn으로
# "처음 보는 test 데이터"에서의 예측 오차(RMSE·MAE)를 잰다.

# --- (1) 설계행렬 만들기 ---
target = "sysBP"
# TODO: 누수 방지 — sysBP(target)만 빼고 나머지를 설명변수 후보로. ____
feature_cols = [c for c in df.columns if ____]

# 결측이 있으면 학습이 안 된다 → 설명변수+목표를 함께 놓고 결측 '행' 제거.
# TODO: 아래에서 결측 행을 통째로 제거하는 메서드를 채워라. ____
design = df[feature_cols + [target]].____

# education만 원핫 인코딩(one-hot), 나머지는 숫자 그대로.
# TODO: education을 원핫하도록 인자를 채워라. ____
X = pd.get_dummies(design[feature_cols], columns=[____], drop_first=True)
y = design[target]

print("사용한 행 수:", len(design), "/ 설명변수 개수:", X.shape[1])

# --- (2) train/test 분할 (재현 위해 random_state=MY_SEED) ---
# TODO: test 비율 0.2, random_state=MY_SEED 로 분할. ____
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=____, random_state=____
)
print("train:", X_train.shape[0], " test:", X_test.shape[0])

# --- (3) 학습은 train에서만 ---
# TODO: train 데이터로만 적합. ____
lin = LinearRegression().fit(____, ____)

# --- (4) 평가는 test에서만 ---
y_pred = lin.predict(X_test)
# TODO: RMSE = sqrt(MSE), MAE 를 test에서 계산. ____
rmse = np.sqrt(metrics.mean_squared_error(____, ____))
mae  = metrics.mean_absolute_error(____, ____)

print(f"[예측 관점] test RMSE = {rmse:.3f} mmHg")
print(f"[예측 관점] test MAE  = {mae:.3f} mmHg")
print(f"RMSE - MAE 간격 = {rmse - mae:.3f}")

# --- (5) 해석 지표(in-sample R²)와 나란히 비교 ---
X_sm = sm.add_constant(X.astype(float))
ols_all = sm.OLS(y, X_sm).fit()
print(f"[해석 관점] in-sample R^2 = {ols_all.rsquared:.3f}")


**해석 질문 (본인이 출력한 실제 값을 인용해 답하세요)**

1. 이번에 나온 **test RMSE = ____ mmHg** 와 **MAE = ____ mmHg** 를 적으세요. 두 값 중 어느 쪽이 항상 더 크며, 그 이유는 무엇인가요? 그리고 둘의 간격($\mathrm{RMSE} - \mathrm{MAE}$)이 알려주는 것은 무엇인가요? (힌트: RMSE는 오차를 제곱합니다 → 크게 빗나간 예측에 어떤 일이 생길까요?)

2. RMSE는 "평균적으로 예측이 약 몇 mmHg 빗나간다"로 대략 읽을 수 있어요. 본인이 얻은 RMSE는 sysBP의 표준편차($\approx 22$ mmHg, 1.1에서 확인)와 비교하면 어느 정도인가요? 이 모델의 예측은 **"그냥 전체 평균으로 찍는 것"보다 확실히 나은가요?**

3. **해석 지표 $R^2$(in-sample)** 와 **예측 지표 test RMSE** 는 서로 **다른 질문**에 답합니다. 각각이 답하는 질문을 한 문장씩으로 쓰세요. (예: $R^2$ → "이 데이터를 얼마나 잘 설명하나?", RMSE → "처음 보는 환자를 얼마나 잘 맞히나?") 이 구분이 리뷰 논문 Shmueli, *"To Explain or to Predict?"* 의 핵심입니다.

4. "$R^2$이 높으면 예측도 잘한다" 는 항상 맞을까요? $R^2$을 **학습에 쓴 그 데이터에서** 재면 왜 예측 성능의 증거가 되기 어려운지, train/test 분할이 왜 필요했는지 본인 말로 설명하세요.

## 2. 스스로 생각하기

여기까지는 "돌리면 나오는" 영역이었습니다. 코드를 실행해 값을 얻는 것은 이제 기본기이고, 진짜 실력은 **그 값을 어떻게 읽고 판단하느냐**에서 갈립니다. 아래 문제들은 정답 코드 한 줄로 끝나지 않습니다. 여러분이 **직접 돌려서 나온 본인의 숫자**를 근거로, 그 숫자가 무엇을 의미하는지, 어디서 함정에 빠질 수 있는지를 스스로 서술해야 합니다. 남의 답을 베끼면 값이 안 맞아 바로 드러나니, 반드시 본인 노트북에서 실행한 결과를 인용하세요.

### 2-A. 흡연 역설 — 연관을 인과로 착각하지 않기

두 변수 사이에 통계적 연관(association)이 있다고 해서 한쪽이 다른 쪽을 일으키는(causation) 것은 아닙니다. 겉보기 연관이 사실은 **제3의 변수** 때문에 생기는 현상을 교란(confounding)이라 부르고, 그 제3의 변수를 교란변수(confounder)라고 합니다. 교란변수는 (1) 원인 후보와도 관련되어 있고 (2) 결과와도 관련되어 있으면서, 두 변수의 관계를 왜곡합니다. 이 문제에서는 흡연 여부(currentSmoker)와 수축기 혈압(sysBP)의 관계를 예로, 단순 비교가 어떻게 사람을 속이는지 직접 확인합니다.

In [ ]:
# === 예측 먼저! (돌리기 전에 한 줄로 적어두세요) ===
# 흡연자와 비흡연자 중 누구의 sysBP 평균이 더 높을 것 같나요? 이유는?
# 내 예측: ___________________________________________

# (1) 단변량 비교: 흡연 여부(currentSmoker)별 sysBP 평균
# TODO: df를 currentSmoker로 groupby 해서 sysBP의 평균과 개수(count)를 출력
uni = df.groupby('currentSmoker')['sysBP'].agg(mean='mean', n='count')
print(uni.round(2))
# TODO: 흡연자(1) - 비흡연자(0) 평균차를 계산해 출력
diff_uni = ____
print(f"흡연자 - 비흡연자 sysBP 평균차 = {diff_uni:.2f} mmHg")

# (2) age를 두 층으로 나눈 뒤, 각 층 안에서 흡연 여부별 sysBP 재비교
age_med = df['age'].median()
df_strat = df.copy()
# TODO: age <= age_med 이면 젊은 층, 아니면 나이든 층으로 age_group 열 만들기 (np.where 사용)
df_strat['age_group'] = np.where(____, 'young', 'old')
# TODO: (age_group, currentSmoker)로 groupby 해서 sysBP 평균 출력
print(____)

# (3) age를 통제한 회귀로 흡연 효과 다시 보기
# TODO: smf.ols 로 "sysBP ~ age + C(currentSmoker)" 적합
m = ____
print(m.summary().tables[1])
# currentSmoker 계수와 p값을 확인하세요 (변수명: C(currentSmoker)[T.1])


아래 네 가지를 **본인이 실행해 나온 숫자를 그대로 인용**하며 서술하세요 (정해진 정답 없음, 관찰과 논리가 채점 대상):

1. **단변량 결과 인용**: (1)에서 흡연자와 비흡연자의 sysBP 평균은 각각 얼마였나요? 평균차(diff_uni)는 몇 mmHg였나요? 이 숫자만 보면 "흡연자가 혈압이 더 (높/낮)다"는 결론이 나오는지 적으세요.

2. **층화·보정 후 변화**: (2)에서 age를 두 층으로 나눴을 때, 각 층 안에서 흡연자-비흡연자 차이는 어떻게 바뀌었나요(본인 값)? (3)에서 age를 통제한 뒤 currentSmoker의 계수와 p값은 얼마였나요(본인 값)? 단변량 때와 비교해 크기와 방향이 어떻게 달라졌는지 서술하세요.

3. **교란변수 지목**: 이 예시에서 교란변수는 무엇이라고 보나요? 그 변수가 (a) 흡연 여부와, (b) 혈압과 각각 어떤 방향으로 관련되어 있어서 단변량 비교를 왜곡했는지, 여러분의 데이터에서 근거를 들어 설명하세요. (힌트: 흡연자와 비흡연자의 평균 나이를 직접 비교해 봐도 좋습니다.)

4. **해석 방어**: 위 결과를 근거로 "흡연이 혈압을 낮춘다"고 결론 내리면 왜 틀린 해석인지 3~4문장으로 반박하세요. 연관(association)과 인과(causation)를 구분해 설명하고, 만약 실제 인과효과를 알고 싶다면 어떤 점을 더 통제하거나 확인해야 할지 한 가지 이상 제안하세요.

### 2-B. 이진 반응에 선형회귀를 쓰면? — 로지스틱이 필요한 이유를 스스로 발견

지금까지 우리는 `sysBP`(연속형)를 선형회귀로 예측했습니다. 그런데 이 데이터의 **진짜 관심사**는 연속형이 아니라 10년 내 심장질환 발병 여부인 `TenYearCHD`(0 또는 1)입니다.

의문: 그러면 `TenYearCHD`도 그냥 `smf.ols`(선형회귀)로 예측하면 안 될까요? 선형회귀는 임의의 실수를 뱉는데, 우리가 원하는 건 "발병 확률"(0~1 사이)입니다. 이 둘이 서로 잘 맞을지, 아니면 어긋날지를 **직접 돌려서** 확인해 봅시다.

아래에서 `age`, `sysBP`, `BMI`로 `TenYearCHD`를 선형회귀한 뒤, 예측값이 실제로 어떤 범위를 갖는지 두 눈으로 관찰합니다. (formula 방식이라 결측이 있는 행은 자동으로 제외되어 적합됩니다.)

In [ ]:
# 2-B. 진짜 관심사 TenYearCHD(0/1)를 '그냥' 선형회귀로 예측해 본다.
# (formula 방식은 결측이 있는 행을 자동으로 제외하고 적합한다.)
m_lpm = smf.ols("TenYearCHD ~ age + sysBP + BMI", data=df).fit()
pred = m_lpm.fittedvalues   # 각 사람에 대한 선형회귀 예측값

# TODO: 예측값의 최솟값과 최댓값을 출력하라.
print("예측값 min :", ____)
print("예측값 max :", ____)

# TODO: 예측값이 0보다 작은 사람 수, 1보다 큰 사람 수를 세어 출력하라.
n_below0 = int((____).sum())
n_above1 = int((____).sum())
print("0보다 작은 예측값을 받은 사람 수 :", n_below0)
print("1보다 큰 예측값을 받은 사람 수 :", n_above1)
print("적합에 실제로 쓰인 관측치 수 :", int(m_lpm.nobs))


**Q2-B. (직접 돌려서 관찰 + 판단)**

**(1) 돌리기 전에 예측 먼저.** 코드를 실행하기 전에, 위 선형회귀의 예측값 중 0~1 범위를 벗어나는 사람이 "있을 것 같다 / 없을 것 같다"를 한 줄로 먼저 적으세요. (맞고 틀리고는 채점 대상이 아닙니다. 예측을 적었는지가 중요합니다.)

**(2) 본인 값 인용.** 이제 실행해서 나온 **본인의 출력값**을 그대로 적으세요.
- 예측값 min = ____ , max = ____
- 0보다 작은 예측값을 받은 사람 수 = ____ 명
- 1보다 큰 예측값을 받은 사람 수 = ____ 명

**(3) 판단.** `TenYearCHD`는 0 또는 1뿐이고, 우리는 이 선형회귀 예측값을 "10년 내 발병 확률"로 해석하고 싶었습니다. 본인이 (2)에서 관찰한 값에 비추어, 이 예측값을 그대로 "확률"이라고 불러도 되는지 판단하고 그 근거를 쓰세요. (특히 min/max와 범위 이탈 인원 수를 근거로 삼으세요. 만약 이탈한 사람이 0명이었다면, 그렇다고 해서 "확률로 써도 안전하다"고 결론지어도 되는지 — age·sysBP·BMI 값이 더 극단적인 새 환자가 들어와도 항상 0~1에 머물 것이라 보장되는지 — 도 함께 논하세요.)

**(4) 본인 언어로.** 위 관찰을 근거로, 이진 반응(0/1)을 예측할 때 선형회귀 대신 **로지스틱 회귀**가 왜 필요한지 한 문단으로 설명하세요. (로지스틱 회귀가 예측값의 범위 문제를 어떻게 다루는지를 본인 말로 연결해서 쓰면 좋습니다.)

### 2-C. 로지스틱 회귀로 진짜 표적을 예측 — Framingham Risk Score를 직접 만듭니다

2-B에서 우리는 이진 결과에 선형회귀를 쓰면 예측값이 확률 범위(0~1)를 벗어난다는 걸 눈으로 확인했습니다. 그 해법이 **로지스틱 회귀**입니다.

큰 그림을 맞춥시다. 강의는 연속형 `sysBP` 회귀에서 멈췄지만, **Framingham 연구가 임상에 실제로 남긴 산물**은 연속값 예측이 아니라 **"앞으로 10년 안에 심장질환이 생길 확률"을 개인별로 계산하는 점수 — Framingham Risk Score** 였습니다. 그 점수의 엔진이 바로 지금 만들 로지스틱 회귀입니다.

강의의 두 절차로 정리하면: **절차①(검정)** 으로 위험인자 후보를 골랐고 **절차②(회귀)** 로 효과의 크기를 쟀습니다. 이제 그 위험인자들을 한데 넣어 **진짜 표적 `TenYearCHD`(0/1)** 의 발병 확률을 예측하고, 각 계수에 `exp()`를 취해 **오즈비(odds ratio)** 로 "이 인자가 위험을 몇 배 올리나"를 읽습니다. 이것이 이 과제가 강의를 이어받아 완성하는 지점입니다.

아래에서 `age·male·sysBP·totChol·currentSmoker·glucose·BMI`로 `TenYearCHD`를 로지스틱 회귀합니다. 계수→오즈비, 예측 확률의 범위(2-B와 대조), `roc_auc_score`로 분류 성능(0.5=무작위, 1.0=완벽)을 읽습니다. 결측은 `dropna`로 제거합니다.

In [ ]:
# 2-C. 로지스틱 회귀 = Framingham Risk Score 엔진.  진짜 표적 TenYearCHD(0/1) 예측.
feat_log = ['age', 'male', 'sysBP', 'totChol', 'currentSmoker', 'glucose', 'BMI']
d_log = df[feat_log + ['TenYearCHD']].____()          # TODO: 결측 제거(dropna)
print("사용 표본 수:", len(d_log), "/ 전체:", len(df))

logit_model = smf.logit(
    "TenYearCHD ~ age + male + sysBP + totChol + currentSmoker + glucose + BMI",
    data=d_log).fit(disp=0)

# TODO: 계수에 np.exp() 를 취해 오즈비(odds_ratio) 열을 만들라.
or_table = pd.DataFrame({
    'coef': logit_model.params,
    'odds_ratio': ____,                               # TODO: np.exp(logit_model.params)
    'p_value': logit_model.pvalues,
})
print(or_table.round(4).to_string())

# 각 사람의 10년 발병 '확률' 예측 (2-B의 선형회귀와 달리 항상 0~1 안에 있는지 확인)
p_hat = logit_model.predict(d_log)
print("\n예측 확률 min / max :", round(float(p_hat.min()), 4), "/", round(float(p_hat.max()), 4))

# TODO: roc_auc_score(정답 라벨, 예측 확률) 로 ROC-AUC 를 구하라.
auc_log = roc_auc_score(d_log['TenYearCHD'], ____)    # TODO: p_hat
print("ROC-AUC :", round(auc_log, 4))

**Q2-C. (본인 출력값 인용 + 판단)**

**(1) 오즈비 읽기.** 출력한 `odds_ratio`에서 **1보다 큰**(위험을 올리는) 변수 2개와, **1에 가깝거나 p값이 유의하지 않은** 변수 1개를 골라 값과 함께 적으세요. 각각에 대해 "이 변수 한 단위 증가가 10년 발병 위험(오즈)을 몇 배로 바꾸는가"를 한 문장으로 해석하세요.

**(2) 범위 문제 해결 확인.** 2-B의 선형회귀 예측값은 0~1을 벗어났습니다. 이번 로지스틱 예측 확률의 min/max는 얼마였나(본인 값)? 2-B와 나란히 놓고, 로지스틱이 "확률"이라는 요구를 어떻게 만족시키는지 본인 숫자로 설명하세요.

**(3) 성능은 하나로 판단하지 않기.** 본인 ROC-AUC는 얼마이며 무작위(0.5)보다 얼마나 나은가? 이 값 하나만으로 "좋은 모델"이라 단정할 수 있는지, 없다면 무엇을 더 봐야 하는지(예: 양성(발병) 비율이 얼마나 불균형한지, 확률 캘리브레이션) 한 가지 이상 적으세요.

**(4) 큰 그림 완성.** 지금 만든 로지스틱이 왜 "Framingham Risk Score의 엔진"인지, 그리고 이 과제가 강의(연속형 `sysBP`에서 멈춤)를 어떻게 **이어서 완성**하는지 2~3문장으로 정리하세요. (절차①=검정 → 절차②=회귀 → 진짜 표적 예측으로 이어지는 흐름을 본인 말로.)

### 2-D. 다중검정의 함정 — 여러 개 유의하면 좋은 건가?

건강검진에서 재는 항목은 한둘이 아닙니다. 나이, 혈압, 콜레스테롤, BMI, 혈당, 심박수... 이걸 전부 CHD 발병군(TenYearCHD==1)과 비발병군(TenYearCHD==0) 사이에서 t검정으로 비교하면, 그중 몇 개는 거의 확실하게 "$p<0.05$, 유의함"으로 나옵니다.

문제는 이것입니다. **검정을 많이 할수록, 실제로는 아무 관련이 없어도 순전히 우연으로 "유의"가 튀어나올 확률이 쌓입니다.** 검정 하나의 유의수준 0.05는 "귀무가설(차이 없음)이 참인데도 유의하다고 잘못 외칠 확률 5%"라는 뜻입니다. 7번 검정하면 그 5%짜리 사고가 7번의 기회를 얻습니다.

아래 셀은 연속변수 7개 각각에 대해 CHD군과 비CHD군의 평균 차이를 Welch t검정(등분산 가정 없음, `equal_var=False`)으로 비교하고, $p<0.05$인 변수가 몇 개인지 셉니다. 결측은 변수마다 따로 `dropna` 처리합니다.

먼저 예측부터 적어 봅시다: 7개 중 몇 개가 유의하게 나올 것 같은가요? 그리고 만약 이 변수들이 CHD와 **아무 관련이 없다면** 우연히 몇 개가 유의하게 나올까요? 돌리기 전에 숫자를 적으세요.

In [ ]:
cont_vars = ['age', 'sysBP', 'diaBP', 'totChol', 'BMI', 'glucose', 'heartRate']

n_sig = 0
for v in cont_vars:
    sub = df[[v, 'TenYearCHD']].dropna()
    g1 = sub.loc[sub['TenYearCHD'] == 1, v]   # CHD 발병군
    g0 = sub.loc[sub['TenYearCHD'] == 0, v]   # 비발병군
    # TODO: Welch t검정 (stats.ttest_ind, equal_var=False) 으로 t, p 구하기
    t, p = ____
    flag = '  <-- 유의 (p<0.05)' if p < 0.05 else ''
    print(f'{v:>10s}:  t = {t:+6.2f},  p = {p:.3e}{flag}')
    # TODO: p < 0.05 이면 n_sig 를 1 증가
    if ____:
        n_sig += 1

print()
print(f'검정한 연속변수 개수      : {len(cont_vars)}')
print(f'유의(p<0.05)한 변수 개수  : {n_sig}')
# TODO: 귀무가설이 모두 참일 때 우연히 기대되는 유의 개수 = 변수 개수 x 0.05
print(f'우연히 기대되는 유의 개수 : {____:.2f}')


**본인 출력값을 인용해 답하세요.**

1. 위에서 유의($p<0.05$)하게 나온 변수는 **몇 개**였나요? 어떤 변수들인지 이름도 함께 적으세요. (본인 셀 출력 그대로)

2. 만약 이 7개 변수가 실제로는 CHD와 **전혀 무관하다면**, 순전히 우연으로 7개 중 몇 개가 유의하게 나올 것으로 기대되나요? (대략 개수 $\times 0.05$로 계산하고 그 값을 적으세요.)

3. 본인 결과의 유의 개수와 2번의 "우연 기대 개수"를 나란히 비교하세요. 이 데이터에서 유의 개수가 우연 기대치보다 훨씬 큰지, 비슷한지 서술하세요.

4. 위 관찰을 근거로, "검정을 여러 개 돌려서 유의한 변수를 여러 개 찾았다"는 주장을 왜 조심해야 하는지 본인 말로 설명하세요. 예를 들어 만약 20개, 100개 변수를 검정했다면 우연히 유의하게 나올 변수는 대략 몇 개일지 언급하고, 이 문제를 줄이려면 어떤 태도(예: 사전 가설 설정, 반복 검증, 다중검정 보정 등)가 필요할지 본인 생각을 덧붙이세요.

(정답이 하나로 정해진 문제가 아닙니다. 본인이 관찰한 숫자에서 출발해 논리적으로 방어하면 됩니다.)

### 2-E. 설명이냐 예측이냐 — 본인 결정 (리뷰 논문 연결)

지금까지 sysBP(수축기 혈압)를 두 방식으로 다뤄 봤습니다. 여러 변수로 sysBP를 "설명"하는 다중회귀($R^2$)와, 학습/평가 데이터를 나눠 새 환자의 sysBP를 "맞히는" 예측 모델(test RMSE)입니다.

여기서는 코드가 주인공이 아닙니다. **본인이 실제로 얻은 두 숫자(다중회귀 $R^2$, test RMSE)를 근거로, 두 가지 서로 다른 목적에 이 모델을 쓸 수 있는지 스스로 판단하고 방어**하는 것이 과제입니다. 정답은 하나가 아니며, "그럴듯한 말"이 아니라 본인 숫자에 붙은 근거를 봅니다.

아래 셀은 판단에 쓸 두 값을 한 번 더 출력할 뿐입니다(이미 값을 알고 있으면 건너뛰어도 됩니다).

In [ ]:
# --- 판단에 쓸 두 숫자 다시 확인 (근거로 인용할 값) ---
# 1) 설명 관점: 다중회귀 R^2
_predictors = ["age", "BMI", "totChol", "glucose", "heartRate", "cigsPerDay"]
_d = df[["sysBP"] + _predictors].dropna()
_mult = smf.ols("sysBP ~ " + " + ".join(_predictors), data=_d).fit()
print(f"[설명] 다중회귀 R^2 = {_mult.rsquared:.3f}   (n = {_d.shape[0]})")

# 2) 예측 관점: test RMSE 를 출력하라 (앞 셀에서 만든 rmse 변수 사용)
# TODO: print(...)
print(f"[예측] test RMSE = {____:.3f} mmHg")

# 이 두 숫자를 아래 서술형 답의 근거로 인용할 것.


**문제 2-E. 이 모델, 어디에 쓸 것인가 — 본인 숫자로 결정하세요**

같은 병원에서 두 부서가 서로 다른 것을 부탁했습니다.

- (A) **설명**: "나이·BMI·콜레스테롤 중 어떤 요인이 혈압과 연관되나? 무엇을 관리하라고 환자에게 말해야 하나?"
- (B) **예측**: "다음 주 올 새 환자의 sysBP 값을 미리 맞히고 싶다. 오차가 얼마나 되나?"

위 셀에서 나온 **본인의 다중회귀 $R^2$**와 **test RMSE 값을 반드시 숫자로 인용**하면서 아래에 답하세요. (정답은 하나가 아닙니다. 근거 있는 결정을 봅니다.)

**(1) 예측 먼저 적기 (돌린 값 보기 전 마음속으로).**
sysBP는 보통 90~180 mmHg 범위를 오갑니다. "새 환자 혈압을 $\pm$몇 mmHg 안에서 맞히면 임상적으로 쓸 만하다"고 생각하는지 본인 기준을 한 문장으로 먼저 적으세요(예: "$\pm 10$ 이내면 유용"). 그 다음 본인 test RMSE와 비교하세요 — 기준을 통과하는가, 못 하는가?

**(2) 목적별 판단.**
- (A) 설명 목적: 본인 $R^2$ 값을 인용하며, 이 모델을 "어떤 요인이 혈압과 연관되는지 말하는 근거"로 쓸 수 있는지 판단하세요. $R^2$가 이 정도일 때 설명용으로 **써도 되는 이유** 또는 **조심해야 하는 이유**를 각각 최소 하나씩 대세요.
- (B) 예측 목적: 본인 test RMSE를 (1)의 본인 기준과 나란히 놓고, 이 모델로 새 환자 혈압을 "맞힌다"고 병원에 내놓을 수 있는지 결정하세요. 결정과 그 한계를 함께 쓰세요.

**(3) 리뷰 논문에 비추어 평가.**
Shmueli의 "To Explain or to Predict?"는 **설명을 잘하는 모델이 반드시 예측을 잘하는 것은 아니다**(설명력과 예측력은 다른 목표이며, 편향-분산 트레이드오프가 개입한다)라고 주장합니다. 본인의 $R^2$와 RMSE 두 숫자가 이 주장에 대해 **무엇을 말해 주는지** 평가하세요. 두 숫자가 이 논지를 뒷받침하는가, 아니면 이 데이터에서는 판단이 애매한가? 본인 관찰에 근거해 방어하세요(교과서 정답을 옮겨 적지 말 것).

## 3. 결론

이번 프로젝트에서 우리는 하나의 데이터(Framingham, n=4,240)를 가지고 **추론 → 회귀 해석 → 예측**이라는 세 가지 통계적 사고의 흐름을 모두 통과했습니다. 이제 각 단계에서 **본인이 직접 뽑은 값**을 근거로 무엇을 알게 되었는지 자기 언어로 정리해 봅시다. (남의 요약이 아니라 내 노트북의 출력 숫자를 인용하세요.)

**(1) 추론으로 알게 된 것 — 신뢰구간 + t검정**

앞에서 부트스트랩으로 만든 `ci_low`, `ci_high`(평균의 95% 신뢰구간)와, `model_simple`(sysBP ~ age)의 회귀계수 t검정 결과(계수 추정치, 표준오차, t값, p-value)를 다시 꺼내 봅시다.

- 내 신뢰구간은 대략 [ (여기에 답: ci_low 값) , (여기에 답: ci_high 값) ] 였고, 이 구간이 의미하는 바를 "만약 같은 방식으로 표본을 여러 번 다시 뽑는다면..." 이라는 문장으로 풀어 쓰면: (여기에 답)
- age 계수의 p-value는 (여기에 답) 이었고, 이걸 근거로 "나이가 수축기혈압과 관련이 있다/없다"를 판단하면: (여기에 답)
- 신뢰구간과 t검정(가설검정)은 사실 같은 정보를 다른 각도에서 본 것입니다. 내가 이해한 그 연결 관계는: (여기에 답)

**(2) 회귀 해석으로 알게 된 것**

`model_simple`의 기울기(age 계수)를 **단위까지 붙여서** 해석해 봅시다.

- 나이가 1살 많아질 때 수축기혈압(sysBP)의 예측 평균은 약 (여기에 답) mmHg 만큼 (증가/감소) 합니다.
- 절편(Intercept)의 숫자는 (여기에 답) 인데, 이 값이 현실적으로 해석 가능한지(나이 0살의 혈압?) 아니면 외삽에 불과한지에 대한 내 판단: (여기에 답)
- 결정계수(R-squared 등)를 보면 age 하나로 sysBP의 변동을 얼마나 설명하는지 알 수 있습니다. 내 값은 (여기에 답) 이고, 이 숫자가 크다/작다에 대한 내 해석과, "관련은 있지만 그것만으로 예측하긴 부족하다" 같은 뉘앙스: (여기에 답)

**(3) 예측 성능으로 알게 된 것**

`lin`(train으로 학습한 선형모델)을 `X_test`에 적용해 얻은 `rmse`, `mae`를 근거로 정리합니다.

- 내 rmse는 (여기에 답), mae는 (여기에 답) 였습니다. 두 값의 단위와, 왜 보통 rmse가 mae보다 큰지(큰 오차에 더 민감)에 대한 내 이해: (여기에 답)
- 이 오차 크기가 실제 혈압 값의 스케일(수십에서 수백 mmHg)에 비추어 "쓸 만한 예측"인지에 대한 내 판단: (여기에 답)
- 추론(2단계)에서 "통계적으로 유의하다"는 것과 예측(3단계)에서 "실제로 잘 맞힌다"는 것이 **같은 말이 아님**을 이번에 몸으로 느꼈다면, 그 차이를 한두 문장으로: (여기에 답)

**(4) 진짜 표적으로 스토리 완성 — 로지스틱**

§2-C에서 이진 표적 `TenYearCHD`를 로지스틱 회귀로 예측하고, 계수를 오즈비로, 성능을 ROC-AUC로 읽었습니다.

- 오즈비가 가장 컸던(위험을 가장 크게 올린) 변수는 (여기에 답)이었고, 그 값과 의미는: (여기에 답)
- 내 ROC-AUC는 (여기에 답)이었습니다. 2-B에서 본 선형회귀의 범위 이탈 문제를 로지스틱이 어떻게 해결했는지(예측 확률 min/max로): (여기에 답)
- 강의는 연속형 `sysBP`에서 멈췄지만, 이 과제는 진짜 표적의 발병 확률 예측(= Framingham Risk Score의 엔진)까지 왔습니다. 절차①(검정) → 절차②(회귀) → 위험 예측으로 이어지는 전체 흐름을 한 문장으로 요약하면: (여기에 답)


## 4. 회고

이번 과제를 하면서 겪은 어려움과 해결 과정을 3~5문장으로 솔직하게 적습니다. 아래 물음을 참고하되 정해진 정답은 없습니다.

- 가장 오래 막혔던 부분은 무엇이었고(예: 결측치 처리, 부트스트랩 반복문, train/test 분리, 계수 해석), 어떻게 풀었나요: (여기에 답)
- 코드는 돌아갔지만 "왜 이렇게 하는지"가 끝까지 헷갈렸던 개념 (신뢰구간의 정확한 의미, p-value의 오개념, RMSE vs MAE, 유의성과 예측력의 차이 등)을 1~3개 골라 적습니다: (여기에 답)
- 다음에 같은 데이터를 다시 분석한다면 이번과 다르게 해보고 싶은 점: (여기에 답)

## 5. 생성형 AI 활용 (비판적 사용)

AI를 **검산기**로 쓰는 것은 권장하지만, 최종 판단은 본인이 내려야 합니다. 아래를 반드시 채우세요. (막연히 "AI 안 썼다"가 아니라, 썼다면 어디에 어떻게 썼는지 구체적으로.)

**(1) 어디에 썼나요**
- AI를 사용한 지점(예: 문법 오류 디버깅, 부트스트랩 개념 설명, 계수 해석 문장 다듬기 등)을 구체적으로: (여기에 답)

**(2) AI가 틀렸거나 이 데이터엔 안 맞는 답을 준 사례** (강조)
- AI가 준 답 중에서 **이번 Framingham 데이터/코드에는 맞지 않았던 것**이 있었나요? 예시로 자주 나오는 오류: 우리 변수 계약(df, model_simple, X_train 등)을 무시하고 변수를 새로 정의하라고 하거나, 존재하지 않는 열 이름을 만들어 내거나, 결측치를 고려하지 않은 코드를 주거나, 실제 내 출력과 다른 숫자를 "결과는 이럴 것"이라고 단정한 경우 등.
- 내가 겪은 사례: (여기에 답 — 없었다면 "왜 없었는지"도 적으세요. 예: 값을 물어보지 않고 코드 문법만 물어봐서)
- 그것이 틀렸다는 걸 **어떻게 알아챘나요**(내 실제 출력값과 대조? 강의 노트와 비교? 직접 다시 돌려봄?): (여기에 답)
- 어떻게 고쳤나요: (여기에 답)

**(3) 실제 대화 내역 붙여넣기**
- AI에게 물어본 프롬프트와 답변의 핵심 부분을 아래에 붙여넣습니다(길면 관련 부분만 발췌). 검산 과정에서 내가 어떤 후속 질문/수정을 했는지가 드러나면 좋습니다.

```
(여기에 AI와의 실제 대화 내역을 붙여넣기)
```

- 마지막으로 한 문장: 이번 과제에서 **AI가 대신할 수 없었던 나만의 판단**은 무엇이었나요? (여기에 답)